In [ ]:
# ----------------------------- Instll Kubeadm ------------------------ # 

In [ ]:
- name: Upgrade system safely
  ansible.builtin.dnf:
    name: "*"
    state: latest
    update_only: yes

- name: Disable swap immediately
  ansible.builtin.command: swapoff -a
  changed_when: false

- name: Disable swap permanently in /etc/fstab
  ansible.builtin.replace:
    path: /etc/fstab
    regexp: '^([^#].*\s+swap\s+.*)$'
    replace: '# \1'


- name: Ensure required kernel modules config file
  copy:
    dest: /etc/modules-load.d/k8s.conf
    content: |
      overlay
      br_netfilter
    owner: root
    group: root
    mode: '0644'

- name: Load overlay module
  community.general.modprobe:
    name: overlay
    state: present

- name: Load br_netfilter module
  community.general.modprobe:
    name: br_netfilter
    state: present

- name: Add Docker CE repository
  ansible.builtin.get_url:
    url: https://download.docker.com/linux/centos/docker-ce.repo
    dest: /etc/yum.repos.d/docker-ce.repo
    mode: '0644'

- name: Set SELinux to permissive
  ansible.posix.selinux:
    policy: targeted
    state: permissive

- name: Disable firewalld
  ansible.builtin.systemd:
    name: firewalld
    enabled: false
    state: stopped

- name: Install Packages
  ansible.builtin.dnf:
    name: 
      - containerd.io
      - containernetworking-plugins
      - tcptraceroute 
      - nc
      - tcpdump
      - vim-enhanced
      - wget
      - unzip
      - git 
      - bind-utils 
      - mtr 
      - tcpdump 
      - nmap-ncat 
      - htop 
      - sysstat 
      - iotop 
      - open-vm-tools 
      - smartmontools 
      - pciutils 
      - usbutils
      - net-tools
      - ipvsadm
    state: present


# - name: Ensure firewalld is installed
#   ansible.builtin.dnf:
#     name: firewalld
#     state: present

# - name: Enable and start firewalld
#   ansible.builtin.systemd:
#     name: firewalld
#     enabled: true
#     state: started
  

# - name: Open Kubernetes control-plane ports
#   ansible.posix.firewalld:
#     port: "{{ item }}"
#     permanent: true
#     state: enabled
#     immediate: true
#   loop:
#     - 6443/tcp
#     - 2379-2380/tcp
#     - 10250/tcp
#     - 10257/tcp
#     - 10259/tcp
#     - 4789/udp
#     - 10250/tcp
#     - 30000-32767/tcp
#     - 4789/udp
#     - 179/tcp
#     - 8472/udp
#     - 80/tcp
#     - 443/tcp
#     - 8081/tcp

# - name: Reload firewalld
#   ansible.builtin.service:
#     name: firewalld
#     state: reloaded


- name: Add the Kubernetes yum repository.
  ansible.builtin.yum_repository:
    name: Kubernetes
    description: Kubernetes
    file: /etc/yum.repos.d/kubernetes.repo
    baseurl: https://pkgs.k8s.io/core:/stable:/v1.35/rpm/
    gpgcheck: true
    gpgkey: https://pkgs.k8s.io/core:/stable:/v1.35/rpm/repodata/repomd.xml.key
    enabled: true
    exclude: kubelet kubeadm kubectl cri-tools kubernetes-cni


- name: Install kubelet, kubeadm , kubectl and containerd
  ansible.builtin.dnf:
    name: 
      - kubelet 
      - kubeadm 
      - kubectl 
    state: present
    disable_excludes: Kubernetes



- name: Copy all CNI binaries to /opt/cni/bin
  shell: cp /usr/libexec/cni/* /opt/cni/bin/
  become: true

- name: Enable the kubelet service before running kubeadm
  ansible.builtin.systemd_service:
    state: started
    name: kubelet
    enabled: true


- name: Enable the containerd service before running kubeadm
  ansible.builtin.systemd_service:
    name: containerd
    enabled: true
    state: started


- name: Create containerd config directory
  ansible.builtin.file:
    path: /etc/containerd
    state: directory
    mode: '0755'


# ------- Kube Network Settings -------

- name: Configure kernel sysctl parameters for Kubernetes
  ansible.posix.sysctl:
    name: "{{ item.key }}"
    value: "{{ item.value }}"
    sysctl_file: /etc/sysctl.d/k8s.conf
    state: present
    reload: true
  loop:
    - { key: 'net.ipv4.ip_forward', value: '1' }
    - { key: 'net.bridge.bridge-nf-call-iptables', value: '1' }
    - { key: 'net.bridge.bridge-nf-call-ip6tables', value: '1' }
    - { key: 'net.ipv4.conf.all.rp_filter', value: '2' }
    - { key: 'net.ipv4.conf.default.rp_filter', value: '2' }

- name: Disable IPv6
  ansible.posix.sysctl:
    name: "{{ item }}"
    value: '1'
    state: present
    reload: true
  loop:
    - net.ipv6.conf.all.disable_ipv6
    - net.ipv6.conf.default.disable_ipv6
  

- name: Generate default containerd config
  ansible.builtin.command:  containerd config default
  register: containerd_config
  changed_when: false

- name: Write containerd config to /etc/containerd/config.toml
  ansible.builtin.copy:
    content: "{{ containerd_config.stdout }}"
    dest: /etc/containerd/config.toml
    owner: root
    group: root
    mode: '0644'

- name: Update SystemdCgroup to true in containerd config
  ansible.builtin.replace:
    path: /etc/containerd/config.toml
    regexp: 'SystemdCgroup = false'
    replace: 'SystemdCgroup = true'
  
- name: Restart containerd
  ansible.builtin.systemd_service:
    name: containerd
    state: restarted

# ----------------------- Auto-Complete

- name: Auto Complete kube
  ansible.builtin.dnf:
    name: bash-completion
    state: present

- name: Add kubectl completion script to .bashrc
  ansible.builtin.lineinfile:
    path: /root/.bashrc
    line: 'source <(kubectl completion bash)'

- name: Apply changes to .bashrc
  ansible.builtin.shell: 'source /root/.bashrc'
  args:
    executable: /bin/bash
  changed_when: false

# ----------------------- reload Daemon
- name: Reload Daemon
  ansible.builtin.systemd_service:
    daemon_reload: true

- name: Ensure python3-pip is installed
  ansible.builtin.package:
    name: python3-pip
    state: present

- name: Install kubernetes python library
  ansible.builtin.pip:
    name: kubernetes
    state: present

In [ ]:
# ------------------------------------------ kub-vip ------------------------------------------ #
# ------------------------------------------ kub-vip ------------------------------------------ #

In [ ]:




cat > kube-vip.sh << 'OF'
#!/bin/bash
# =============================================================================
# Kube-VIP Static Pod Setup Script
# Purpose: Deploy Kube-VIP as a static pod for HA Kubernetes control plane
# Author: Generated for Khaled
# =============================================================================

set -euo pipefail  # Best practice: exit on error, undefined vars, pipe failures

# ========================= CONFIGURATION =========================
VIP="172.16.6.85"
INTERFACE="ens192"
POD_NETWORK_CIDR="10.244.0.0/16"

# Colors for better output
RED='\033[0;31m'
GREEN='\033[0;32m'
YELLOW='\033[1;33m'
BLUE='\033[0;34m'
NC='\033[0m' # No Color

log() {
    echo -e "${BLUE}[INFO]${NC} $1"
}

success() {
    echo -e "${GREEN}[SUCCESS]${NC} $1"
}

warn() {
    echo -e "${YELLOW}[WARN]${NC} $1"
}

error() {
    echo -e "${RED}[ERROR]${NC} $1" >&2
    exit 1
}

# Check if running as root
if [[ $EUID -ne 0 ]]; then
    error "This script must be run as root"
fi

log "Starting Kube-VIP + kubeadm initialization setup..."

# ========================= GET LATEST KUBE-VIP VERSION =========================
log "Fetching latest kube-vip version..."
KVVERSION=$(curl -sL https://api.github.com/repos/kube-vip/kube-vip/releases/latest | \
            jq -r '.tag_name' || echo "v0.8.0")

if [[ -z "$KVVERSION" || "$KVVERSION" == "null" ]]; then
    warn "Failed to fetch latest version, using fallback v0.8.0"
    KVVERSION="v0.8.0"
fi

success "Using kube-vip version: $KVVERSION"

# ========================= SETUP KUBE-VIP ALIAS =========================
log "Setting up kube-vip command..."

# Create a proper wrapper script with version embedded
cat > /usr/local/bin/kube-vip << EOF
#!/bin/bash
# Kube-VIP wrapper - Version: ${KVVERSION}

KVVERSION="${KVVERSION}"
IMAGE="ghcr.io/kube-vip/kube-vip:\${KVVERSION}"

# Pull image if not present
if ! ctr images ls | grep -q "kube-vip:\${KVVERSION}"; then
    echo "Pulling kube-vip image: \${IMAGE}" >&2
    ctr image pull "\${IMAGE}" >/dev/null 2>&1 || {
        echo "Failed to pull kube-vip image" >&2
        exit 1
    }
fi

ctr run --rm --net-host "\${IMAGE}" vip /kube-vip "\$@"
EOF

chmod +x /usr/local/bin/kube-vip

success "kube-vip command installed (version ${KVVERSION})"


# ========================= CREATE KUBE-VIP MANIFEST =========================
log "Generating Kube-VIP static pod manifest..."

mkdir -p /etc/kubernetes/manifests/

kube-vip manifest pod \
    --interface "$INTERFACE" \
    --address "$VIP" \
    --controlplane \
    --services \
    --arp \
    --leaderElection | tee /etc/kubernetes/manifests/kube-vip.yaml

if [[ ! -f /etc/kubernetes/manifests/kube-vip.yaml ]]; then
    error "Failed to create kube-vip manifest"
fi

success "Kube-VIP manifest created at /etc/kubernetes/manifests/kube-vip.yaml"

# ========================= VERIFY IMAGE =========================
log "Verifying kube-vip image..."
ctr images ls | grep -i kube-vip || warn "kube-vip image not found in containerd"

log "Starting Kubernetes cluster initialization..."

echo ""
echo "═══════════════════════════════════════════════════════════════"
echo "                  🚀 RUNNING KUBEADM INIT"
echo "═══════════════════════════════════════════════════════════════"
echo ""

kubeadm init \
    --control-plane-endpoint "${VIP}:6443" \
    --upload-certs \
    --pod-network-cidr "$POD_NETWORK_CIDR" \
    --apiserver-cert-extra-sans "$VIP" 2>&1 | tee /var/log/kubeadm-init-$(date +%Y%m%d-%H%M).log

INIT_EXIT_CODE=${PIPESTATUS[0]}

if [[ $INIT_EXIT_CODE -eq 0 ]]; then
    success "✅ Kubernetes control plane initialized successfully!"
    
    # Extract and show join commands clearly
    echo ""
    echo "📋 IMPORTANT JOIN COMMANDS:"
    echo "───────────────────────────────────────────────────────────────"
    grep -A 5 -E "(kubeadm join|certificate-key)" /var/log/kubeadm-init-*.log | tail -n 20
    echo "───────────────────────────────────────────────────────────────"
else
    error "❌ kubeadm init failed with exit code $INIT_EXIT_CODE"
fi

# ========================= SETUP KUBECONFIG =========================
log "Setting up kubectl configuration for current user..."

mkdir -p $HOME/.kube
cp -i /etc/kubernetes/admin.conf $HOME/.kube/config
chown $(id -u):$(id -g) $HOME/.kube/config

success "✅ Kubeconfig configured successfully!"
echo "You can now run: kubectl get nodes"

# ========================= FINAL SUMMARY =========================
echo ""
echo "═══════════════════════════════════════════════════════════════"
echo "🎉 Setup Completed Successfully!"
echo "═══════════════════════════════════════════════════════════════"
echo "VIP Address      : $VIP"
echo "Interface        : $INTERFACE"
echo "Kube-VIP Version : $KVVERSION"
echo "Pod CIDR         : $POD_NETWORK_CIDR"
echo ""
echo "Next steps:"
echo "   1. Install your CNI (Flannel, Calico, Cilium, etc.)"
echo "   2. Join other control plane nodes using the join command above"
echo "   3. kubectl get pods -n kube-system -w"
echo "═══════════════════════════════════════════════════════════════"
OF

chmod +x kube-vip.sh && ./kube-vip.sh


In [ ]:
# --------------------------------------- Flannel  --------------------------------------- #
# --------------------------------------- Flannel  --------------------------------------- #


In [ ]:
kubectl apply -f https://github.com/flannel-io/flannel/releases/latest/download/kube-flannel.yml

# --------------------------------------- metalb  --------------------------------------- #
# --------------------------------------- metalb  --------------------------------------- #

In [ ]:
kubectl apply -f https://raw.githubusercontent.com/metallb/metallb/v0.15.3/config/manifests/metallb-native.yaml

In [ ]:
# kubectl delete validatingwebhookconfiguration metallb-webhook-configuration

In [ ]:
cat > metallb-pool.yaml << 'OF'
apiVersion: metallb.io/v1beta1
kind: IPAddressPool
metadata:
  name: default-pool
  namespace: metallb-system
spec:
  addresses:
  - 172.16.6.90-172.16.6.95
---
apiVersion: metallb.io/v1beta1
kind: L2Advertisement
metadata:
  name: l2adv
  namespace: metallb-system
OF

In [ ]:
kubectl apply -f metallb-pool.yaml

In [ ]:
kubectl edit configmap kube-proxy -n kube-system

In [ ]:
strictARP: true

In [ ]:
kubectl rollout restart ds kube-proxy -n kube-system

Ingress

In [ ]:
helm repo add ingress-nginx https://kubernetes.github.io/ingress-nginx
helm repo update

In [ ]:
cat > nginx-ingress-values.yaml <<EOF
controller:
  kind: DaemonSet          # ← change from Deployment to DaemonSet
  hostNetwork: true
  dnsPolicy: ClusterFirstWithHostNet
  hostPort:
    enabled: true
    ports:
      http: 80
      https: 443
  service:
    type: LoadBalancer
    loadBalancerIP: 172.16.6.90
  admissionWebhooks:
    enabled: false
  metrics:
    enabled: true
  podDisruptionBudget:
    enabled: true
  tolerations:             # ← add this to run on master nodes
  - key: "node-role.kubernetes.io/control-plane"
    operator: "Exists"
    effect: "NoSchedule"
EOF

In [ ]:
helm upgrade --install ingress-nginx ingress-nginx/ingress-nginx \
  --namespace ingress-nginx \
  --create-namespace \
  -f nginx-ingress-values.yaml

headlamp

In [ ]:
# 1. Add the official Helm repo
helm repo add headlamp https://kubernetes-sigs.github.io/headlamp/
helm repo update

In [ ]:
cat > headlamp-values.yaml << EOF
ingress:
  enabled: true
  ingressClassName: nginx   
  hosts:
    - host: headlamp.voip.local
      paths:
        - path: /
          type: Prefix
EOF

In [ ]:
helm install my-headlamp headlamp/headlamp \
	-f headlamp-values.yaml \
	--namespace headlamp \
	--create-namespace